In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Type, Tuple
from pathlib import Path
import logging
import pytorch_lightning as pl
from omegaconf import OmegaConf, DictConfig
from oat.predictor_module.oat_predictor import OATPredictor
from oat.data_module.setup import setup_data_and_transform
from matplotlib import pyplot as plt
import torch

def load_checkpoint(predictor_cls: Type[pl.LightningModule], checkpoint_file_path: str)-> Tuple[pl.LightningModule, DictConfig]:
    """
    Load and return the Predictor and its config from a checkpoint file.
    
    predictor_cls is the class of the Predictor to load. 
    checkpoint_file_path is the path to the checkpoint file, not the folder.
    It is expected that the parent folder of this checkpoint contains, at the same 
    level as the checkpoint, a yaml file with the config used to train the model.
    """
    path = Path(checkpoint_file_path)
    checkpoint_folder = path.parent
    config_path = checkpoint_folder / "config.yaml"
    
    cfg = OmegaConf.load(config_path)
    
    predictor = predictor_cls.load_from_checkpoint(path)
    
    return predictor, cfg

## Load model

In [ ]:
cfg_path = "/scratch/mch/cdonner/observation_assimilation_transformer/checkpoints/debugging/oat_training_20260224_job3955558/best-model-epoch=02-val_loss=-0.4826.ckpt"
#cfg_path = "/users/miccatta/s-ruc/observation_assimilation_transformer/checkpoints/lost_checkpoints/good_uninformed/best-model-epoch=26-val_loss=-1.1953.ckpt"
predictor, cfg = load_checkpoint(OATPredictor, cfg_path)
predictor.eval()

## Load data

In [ ]:
logger = logging.getLogger(__name__)
dataset, train_loader, val_loader, test_loader, input_transform, output_transform = setup_data_and_transform(cfg, logger)

In [ ]:
batch = next(iter(test_loader))
prediction = predictor.predict_step(batch, 0)

# Create query around station

In [ ]:
batch_size = cfg.dataloader.batch_size
station_idx = 4
center_pos = batch['query_data']['coords'][0, station_idx]
center_geo = batch['query_data']['geo'][0, station_idx]
size = 10000 # in metres
step = 100 # in metres
extent = [[center_pos[0] - size / 2, center_pos[0] + size / 2], 
          [center_pos[1] - size / 2, center_pos[1] + size / 2]]
mesh = torch.meshgrid(torch.linspace(extent[0][0], extent[0][1], size // step),
                      torch.linspace(extent[1][0], extent[1][1], size // step))
mesh = torch.stack(mesh, dim=-1)
mesh_shape = mesh.shape
mesh_flattened = mesh.view(-1, 2)

query_mesh = dataset.get_query(mesh_flattened)
query_batch = {k: v[None].expand(batch_size, -1, *v.shape[1:]) for k, v in query_mesh.items()}
print(val_loader.dataset.station_info)

In [ ]:
batch['query_data'] = query_batch
predictor.model.num_samples = 2000
prediction = predictor.predict_step(batch, 0)

In [ ]:
mean_pred_mesh = prediction.detach().mean(dim=-2).cpu().numpy().reshape(batch_size, *mesh_shape)

# Plot predictions

In [ ]:
fig = plt.figure(figsize=(10, 5))
var_idx = 0
ax1 = fig.add_subplot(1, 2, 1)
im1 = ax1.pcolor(mesh[:, :, 0], mesh[:, :, 1], mean_pred_mesh[1, :, :, var_idx], shading='auto')
plt.colorbar(im1, label=predictor.output_var_names[var_idx])
ax1.set_xlabel("Swiss Easting")
ax1.set_ylabel("Swiss Northing")
var_idx = 1
ax2 = fig.add_subplot(1, 2, 2)
im2 = ax2.pcolor(mesh[:, :, 0], mesh[:, :, 1], mean_pred_mesh[0, :, :, var_idx], shading='auto')
plt.colorbar(im2, label=predictor.output_var_names[var_idx])
ax2.set_xlabel("Swiss Easting")
ax2.set_ylabel("Swiss Northing")
plt.tight_layout()

# Plot output at different model stages

### Topographic encoder

In [ ]:
with torch.no_grad():
    topo_enc = predictor.model.topographic_encoder(batch['query_data']['topo'])
topo_enc_resh = topo_enc.reshape(batch_size, *mesh_shape[:-1], -1).detach().cpu().numpy()
plt.pcolor(mesh[:, :, 0], mesh[:, :, 1], topo_enc_resh[0, :, :, 0], shading='auto')

### Global Encoder

In [ ]:
with torch.no_grad():
    global_enc = predictor.model.global_encoder(batch['global_data'], batch['query_data'])
global_enc_resh = global_enc.reshape(batch_size, *mesh_shape[:-1], -1).detach().cpu().numpy()
plt.pcolor(mesh[:, :, 0], mesh[:, :, 1], global_enc_resh[0, :, :, 1], shading='auto')

# After first fusion layer

In [ ]:
predictor.eval()
with torch.no_grad():
    cond = batch['global_data']['time_emb'].unsqueeze(1).expand(batch_size, mesh_flattened.shape[0], -1)
    uninformed_token = predictor.model.fusion_layer1(
        global_enc, 
        topo_enc, 
        cond=cond)
uninformed_token_resh = uninformed_token.reshape(batch_size, *mesh_shape[:-1], -1).detach().cpu().numpy()
plt.pcolor(mesh[:, :, 0], mesh[:, :, 1], uninformed_token_resh[0, :, :, 0], shading='auto')
plt.colorbar(label="Uninformed token value")

### Attention bias

In [ ]:
geo_transform = predictor.input_transform.transform_dict['query_data'].transform_dict['geo']
d_geo = torch.abs(geo_transform(center_geo) - geo_transform(query_batch['geo'][0]))[None]
attn_bias = predictor.model.oat_block.decoder.layers[0].self_attn.f_geo(d_geo).detach().cpu()

In [ ]:
attn_bias_mesh = attn_bias.view(*mesh_shape[:-1], -1)

In [ ]:
plt.figure(figsize=(20, 5))
for ihead in range(attn_bias_mesh.shape[-1]):
    plt.subplot(1, attn_bias_mesh.shape[-1], ihead + 1)
    im = plt.pcolor(mesh[:, :, 0], mesh[:, :, 1], attn_bias_mesh[:, :, ihead], shading='auto')
    plt.colorbar(im, label="Attention bias")
    plt.xlabel("Swiss Easting")
    plt.ylabel("Swiss Northing")
    plt.title(f"Attention bias for attention head {ihead}")
    plt.tight_layout()
plt.show()
#plt.tight_layout()

# Geo variables

In [ ]:
# plot d_geo
d_geo_mesh = d_geo.view(*mesh_shape[:-1], -1)
for ivar in range(d_geo_mesh.shape[-1]):
    im = plt.pcolor(mesh[:, :, 0], mesh[:, :, 1], d_geo_mesh[:, :, ivar], shading='auto')
    plt.colorbar(im, label=f"d_geo variable {ivar}")
    plt.xlabel("Swiss Easting")
    plt.ylabel("Swiss Northing")
    plt.title(f"d_geo for variable {ivar}")
    plt.tight_layout()
    plt.show()

In [ ]:
# plot query_batch['geo'][0]
geo_batch_mesh = query_batch['geo'][0].view(*mesh_shape[:-1], -1)

for ivar in range(query_batch['geo'].shape[-1]):
    plt.figure(figsize=(5, 5))
    im = plt.pcolor(mesh[:, :, 0], mesh[:, :, 1], geo_batch_mesh[:, :, ivar], shading='auto')
    plt.colorbar(im, label=f"query geo variable {ivar}")
    plt.xlabel("Swiss Easting")
    plt.ylabel("Swiss Northing")
    plt.title(f"Query geo variable {ivar}")
    plt.tight_layout()
    plt.show()

In [ ]:
query_batch['geo'].shape

In [ ]:
dataset.variable_names